# 02 — Encoding and decoding (Part C, task 3)

Exercises: the Transformers tokenizer API you will use in every LLM project — `tokenize`, `convert_tokens_to_ids`, `encode`, `decode`, calling the tokenizer on a batch.

In [ ]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
%load_ext autoreload
%autoreload 2

from src.pretrained import load_pretrained
pretrained = load_pretrained()     # provided helper: {"gpt2": ..., "bert-base-uncased": ..., "google/flan-t5-small": ...}
assert pretrained, "No pretrained tokenizer could be loaded: check your internet connection."

In [ ]:
tok = pretrained["bert-base-uncased"]        # we work with BERT's tokenizer for the exercises
s = "Hello, how are you?"

### 🧩 Exercise 2.1 — text → tokens → ids → tokens → text

Do the full round trip on the sentence `s`.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `tok.tokenize(text)` → list of token **strings**.
* `tok.convert_tokens_to_ids(tokens)` → list of ints; `tok.convert_ids_to_tokens(ids)` goes back.
* `tok.decode(ids)` → a string (try `skip_special_tokens=True` later).
* Docs: https://huggingface.co/docs/transformers/main_classes/tokenizer

In [ ]:
tokens = tok.____(s)
ids    = tok.____(tokens)
back   = tok.____(ids)
text   = tok.____(ids)

print("tokens :", tokens); print("ids    :", ids); print("back   :", back); print("decoded:", repr(text))

In [ ]:
assert isinstance(tokens, list) and all(isinstance(t, str) for t in tokens)
assert len(ids) == len(tokens) and all(isinstance(i, int) for i in ids)
assert back == tokens
assert isinstance(text, str)
print("✅ Exercise 2.1 OK")

### 🧩 Exercise 2.2 — `encode` and special tokens

Compute the ids with `encode` **with** and **without** the special tokens the tokenizer adds by default, and compare lengths.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `tok.encode(text)` adds special tokens by default (for BERT: where?).
* Switch them off with the argument `add_special_tokens=False`.
* Predict first: how many more ids will the default version contain?

In [ ]:
ids_default = tok.____(s)
ids_plain   = tok.encode(s, ____)

print(len(ids_default), len(ids_plain))
print(tok.convert_ids_to_tokens(ids_default))

In [ ]:
assert ids_plain == tok.convert_tokens_to_ids(tok.tokenize(s))
assert len(ids_default) >= len(ids_plain)
print("✅ Exercise 2.2 OK  -- remember this when you benchmark tokenizers in notebook 07")

### 🧩 Exercise 2.3 — Batches, padding and attention masks

Call the tokenizer **directly** on a list of two sentences of different lengths, padding them to the same length.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `tok(list_of_texts, padding=True)` returns a dictionary-like `BatchEncoding`.
* Keys of interest: `input_ids`, `attention_mask`. The mask is 1 for real tokens and 0 for padding.
* Option `truncation=True, max_length=...` cuts long inputs.

In [ ]:
batch = tok(____, ____)

print(batch["input_ids"])
print(batch["attention_mask"])

In [ ]:
lens = {len(x) for x in batch["input_ids"]}
assert len(lens) == 1, "padding=True must give equal lengths"
assert 0 in batch["attention_mask"][0] and set(batch["attention_mask"][1]) == {1}
print("✅ Exercise 2.3 OK")

### The three tokenizers on the five assignment sentences

In [ ]:
sentences = [
    "Hello, how are you?",
    "Large Language Models are fascinating.",
    "Tokenization is essential for LLMs.",
    "The model is learning from data.",
    "I am studying artificial intelligence.",
]
for model_name, t in pretrained.items():
    print("\nMODEL:", model_name)
    for sent in sentences:
        tokens = t.tokenize(sent)
        ids = t.convert_tokens_to_ids(tokens)
        decoded = t.decode(ids)
        print("Original :", sent); print("Tokens   :", tokens); print("IDs      :", ids)
        print("Decoded  :", decoded, "| identical:", decoded == sent)
        print("-" * 50)

**Questions**
1. `tokenize()` vs `encode()`? 2. `convert_tokens_to_ids()` vs `encode()`?
3. Why do tokenizers give different ids for the same text?
4. Why can decoding give slightly different formatting? List every sentence/tokenizer pair where `decoded != original` and explain **each** difference (which pipeline component is responsible?).
5. What happens with a token that does not exist in the vocabulary? Run the cell below.

**Your answer:**

_(write here)_

In [ ]:
for text in ["日本語のテキスト", "I like pizza 🍕", "الذكاء الاصطناعي"]:
    for name, t in pretrained.items():
        toks = t.tokenize(text)
        print(f"{name:22} {len(toks):3} tokens  unk={sum(x == t.unk_token for x in toks):2}  {toks[:8]}")